# A soil organic carbon degradation boundary for India

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
import pyarrow.parquet as pq
from scipy import stats
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_DIR = "data"
OUT_DIR = "outputs"
import os
os.makedirs(OUT_DIR, exist_ok=True)

RESPONSES = ["BD", "pH", "CEC", "NPP"]
RESPONSE_FULL = {
    "BD": "Physical (Bulk Density)",
    "pH": "Chemical (pH)",
    "CEC": "Chemical (CEC)",
    "NPP": "Ecosystem (NPP)",
}
Z95 = 1.959964
KM_PER_DEG = 111.32

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
print("Setup complete.")

Setup complete.


## 2. Historical SOC baseline

In [2]:
NODATA = -9999.0

with rasterio.open(f"{DATA_DIR}/step13_soc_stack.tif") as src:
    soc_arr = src.read()  # bands: SOC_NoLU, SOC_900, SOC_1800, SOC_1910, SOC_1960, SOC_1990, SOC_2010
    ph1_transform = src.transform

with rasterio.open(f"{DATA_DIR}/step10_feature_stack.tif") as src:
    feat_arr = src.read()

valid_mask_2d = ~np.any(feat_arr == NODATA, axis=0)
rows_idx, cols_idx = np.where(valid_mask_2d)
n_valid = len(rows_idx)

soc_nolu = soc_arr[0][rows_idx, cols_idx]
soc_2010 = soc_arr[6][rows_idx, cols_idx]
F_2010 = soc_2010 / soc_nolu
D_2010_pct = 100.0 * (1.0 - F_2010)

lons = ph1_transform.c + (cols_idx + 0.5) * ph1_transform.a
lats = ph1_transform.f + (rows_idx + 0.5) * ph1_transform.e
cellsize_deg = ph1_transform.a
area_km2 = (cellsize_deg * KM_PER_DEG) * (cellsize_deg * KM_PER_DEG * np.cos(np.radians(lats)))

with rasterio.open(f"{DATA_DIR}/ecoregion_id.tif") as src:
    eco_full = src.read(1)
    eco_transform = src.transform
    eco_crs = src.crs
    eco_nodata = src.nodata

eco_on_ph1 = np.zeros(valid_mask_2d.shape, dtype=np.int32)
reproject(
    source=eco_full, destination=eco_on_ph1,
    src_transform=eco_transform, src_crs=eco_crs,
    dst_transform=ph1_transform, dst_crs=eco_crs,
    src_nodata=eco_nodata, dst_nodata=eco_nodata,
    resampling=Resampling.mode,
)
ecoregion_id = eco_on_ph1[rows_idx, cols_idx]

hist_px = pd.DataFrame({
    "lon": lons, "lat": lats, "Ecoregion_id": ecoregion_id, "area_km2": area_km2,
    "SOC_NoLU_MgCha": soc_nolu, "SOC_2010_MgCha": soc_2010,
    "F_2010": F_2010, "D_2010_pct": D_2010_pct,
})
hist_px = hist_px[hist_px["Ecoregion_id"] > 0].reset_index(drop=True)

print(f"Valid historical grid cells: {n_valid}; assigned to an ecoregion: {len(hist_px)}")
print(f"National (area-weighted) mean F_2010: {np.average(hist_px.F_2010, weights=hist_px.area_km2):.3f}")
print(f"National (area-weighted) mean D_2010: {np.average(hist_px.D_2010_pct, weights=hist_px.area_km2):.2f}%")

Valid historical grid cells: 41507; assigned to an ecoregion: 41507
National (area-weighted) mean F_2010: 0.908
National (area-weighted) mean D_2010: 9.20%


In [3]:
ecu_support = pd.read_csv(f"{DATA_DIR}/ecu_support_table.csv")
eco_lookup = ecu_support[["ecoregion_id", "Ecoregion"]].drop_duplicates().rename(columns={"ecoregion_id": "Ecoregion_id"})
hist_px = hist_px.merge(eco_lookup, on="Ecoregion_id", how="left")

hist_eco = hist_px.groupby(["Ecoregion_id", "Ecoregion"]).apply(
    lambda g: pd.Series({
        "n_cells": len(g),
        "area_km2": g["area_km2"].sum(),
        "F_2010_mean": np.average(g["F_2010"], weights=g["area_km2"]),
        "D_2010_pct": np.average(g["D_2010_pct"], weights=g["area_km2"]),
    }), include_groups=False
).reset_index()

hist_eco.to_csv(f"{OUT_DIR}/historical_baseline_ecoregion.csv", index=False)
print(f"Ecoregion-level historical baseline: {hist_eco.shape}")
hist_eco.sort_values("D_2010_pct", ascending=False).head(5)

Ecoregion-level historical baseline: (48, 6)


,Ecoregion_id,Ecoregion,n_cells,area_km2,F_2010_mean,D_2010_pct
3,5,Central Tibetan Plateau alpine steppe,183.0,12934.035113,0.782212,21.778803
24,27,North Tibetan Plateau-Kunlun Mountains alpine ...,173.0,12125.559081,0.818109,18.189134
44,47,Upper Gangetic Plains moist deciduous forests,3446.0,264071.202363,0.844494,15.550570
7,9,East Deccan dry-evergreen forests,301.0,25338.464752,0.868066,13.193415
33,36,Orissa semi-evergreen forests,268.0,21633.031107,0.876330,12.366973


## 3. Contemporary SOC and empirically-derived functional thresholds

In [4]:
step11 = pd.read_csv(f"{DATA_DIR}/step11_thresholds.csv")
ecu_thresholds = pd.read_csv(f"{DATA_DIR}/step5_threshold_uncertainty.csv")

print("Hierarchical threshold estimates (national / pooled row shown for Physical; ecoregion-level otherwise):")
print(step11[step11["Response"] == "Physical (Bulk Density)"][["Response", "Context", "Threshold_SOC_gC_kg", "CI95_lower_gC_kg", "CI95_upper_gC_kg"]])
print()
for r in RESPONSES:
    sub = ecu_thresholds[ecu_thresholds["Response"] == RESPONSE_FULL[r]]
    print(f"{RESPONSE_FULL[r]:28s} tiered threshold across {len(sub)} ECUs: "
          f"median={sub['Assigned_threshold_gC_kg'].median():.2f}, "
          f"IQR=({sub['Assigned_threshold_gC_kg'].quantile(.25):.2f}, {sub['Assigned_threshold_gC_kg'].quantile(.75):.2f}) g C/kg")

Hierarchical threshold estimates (national / pooled row shown for Physical; ecoregion-level otherwise):
                  Response                                            Context  Threshold_SOC_gC_kg  CI95_lower_gC_kg  CI95_upper_gC_kg
0  Physical (Bulk Density)  Global (pooled - not identified per-Ecoregion,...            47.731197         45.420668         49.301512

Physical (Bulk Density)      tiered threshold across 1156 ECUs: median=19.17, IQR=(12.75, 33.02) g C/kg
Chemical (pH)                tiered threshold across 1156 ECUs: median=18.36, IQR=(13.56, 27.54) g C/kg
Chemical (CEC)               tiered threshold across 1156 ECUs: median=17.84, IQR=(12.63, 24.87) g C/kg
Ecosystem (NPP)              tiered threshold across 1156 ECUs: median=19.79, IQR=(14.32, 30.90) g C/kg


In [5]:
cols = ["ECU_ID", "Ecoregion_id", "Ecoregion", "SoilOrder", "LULC", "SOC_raw"]
tbl = pq.read_table(f"{DATA_DIR}/master_analysis_database.parquet", columns=cols)
pixels = tbl.to_pandas()
del tbl
print(f"Contemporary pixel population: {len(pixels):,} rows")

INV_RESPONSE = {v: k for k, v in RESPONSE_FULL.items()}
thr_wide = ecu_thresholds.pivot_table(index="ECU_ID", columns="Response", values="Assigned_threshold_gC_kg")
thr_wide.columns = [f"T_{INV_RESPONSE[c]}" for c in thr_wide.columns]
pixels = pixels.merge(thr_wide.reset_index(), on="ECU_ID", how="left")
print(pixels[[f"T_{r}" for r in RESPONSES]].describe().loc[["mean", "50%"]])

Contemporary pixel population: 50,475,295 rows


           T_BD       T_pH      T_CEC      T_NPP
mean  14.648135  14.142898  13.987293  16.021403
50%   12.172360  12.833332  12.601584  13.971222


## 4. Literature-referenced threshold comparison

In [6]:
LITERATURE_THRESHOLDS = {"2.0% SOC (20 g/kg)": 20.0, "1.1% SOC (11 g/kg)": 11.0}

for label, val in LITERATURE_THRESHOLDS.items():
    print(f"\n{label} = {val} g C/kg -- position relative to empirically-assigned thresholds:")
    for r in RESPONSES:
        sub = ecu_thresholds[ecu_thresholds["Response"] == RESPONSE_FULL[r]]["Assigned_threshold_gC_kg"]
        pct_rank = 100 * (sub < val).mean()
        print(f"  {RESPONSE_FULL[r]:28s} literature value falls at the {pct_rank:5.1f}th percentile "
              f"of the empirical threshold distribution (median empirical={sub.median():.2f})")


2.0% SOC (20 g/kg) = 20.0 g C/kg -- position relative to empirically-assigned thresholds:
  Physical (Bulk Density)      literature value falls at the  52.8th percentile of the empirical threshold distribution (median empirical=19.17)
  Chemical (pH)                literature value falls at the  56.1th percentile of the empirical threshold distribution (median empirical=18.36)
  Chemical (CEC)               literature value falls at the  56.3th percentile of the empirical threshold distribution (median empirical=17.84)
  Ecosystem (NPP)              literature value falls at the  50.2th percentile of the empirical threshold distribution (median empirical=19.79)

1.1% SOC (11 g/kg) = 11.0 g C/kg -- position relative to empirically-assigned thresholds:
  Physical (Bulk Density)      literature value falls at the  18.5th percentile of the empirical threshold distribution (median empirical=19.17)
  Chemical (pH)                literature value falls at the  15.1th percentile of the empiri

In [7]:
national_rows = []
for r in RESPONSES:
    empirical_below = 100 * (pixels["SOC_raw"] < pixels[f"T_{r}"]).mean()
    row = {"Response": RESPONSE_FULL[r], "Empirical (tiered) threshold, % below": empirical_below}
    for label, val in LITERATURE_THRESHOLDS.items():
        row[f"{label}, % below"] = 100 * (pixels["SOC_raw"] < val).mean()
    national_rows.append(row)

threshold_comparison = pd.DataFrame(national_rows)
threshold_comparison.to_csv(f"{OUT_DIR}/literature_threshold_comparison.csv", index=False)
threshold_comparison

,Response,"Empirical (tiered) threshold, % below","2.0% SOC (20 g/kg), % below","1.1% SOC (11 g/kg), % below"
0,Physical (Bulk Density),55.812318,78.912395,47.261614
1,Chemical (pH),50.851412,78.912395,47.261614
2,Chemical (CEC),49.696338,78.912395,47.261614
3,Ecosystem (NPP),64.869194,78.912395,47.261614


In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(RESPONSES))
width = 0.25
cols_to_plot = ["Empirical (tiered) threshold, % below"] + [f"{l}, % below" for l in LITERATURE_THRESHOLDS]
colors = ["#1565c0", "#f9a825", "#c62828"]
for i, col in enumerate(cols_to_plot):
    ax.bar(x + (i - 1) * width, threshold_comparison[col], width, label=col, color=colors[i])
ax.set_xticks(x)
ax.set_xticklabels([RESPONSE_FULL[r] for r in RESPONSES], rotation=15, ha="right")
ax.set_ylabel("% of India's mapped land area below threshold")
ax.set_title("Empirical, function-derived thresholds vs. literature-cited universal thresholds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_literature_threshold_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Degradation boundary: empirical versus literature-threshold definitions

In [9]:
ecu_thresholds["sd_lnT"] = (np.log(ecu_thresholds["T_U"]) - np.log(ecu_thresholds["T_L"])) / (2 * Z95)

def compute_ecoregion_P(threshold_value=None):
    out = {}
    for r in RESPONSES:
        sub = ecu_thresholds[ecu_thresholds["Response"] == RESPONSE_FULL[r]]
        lut = sub.set_index("ECU_ID")[["T_M", "sd_lnT"]]
        merged = pixels[["ECU_ID", "SOC_raw", "Ecoregion_id"]].merge(
            lut, left_on="ECU_ID", right_index=True, how="left")
        if threshold_value is not None:
            merged["P_below"] = (merged["SOC_raw"] < threshold_value).astype(float)
        else:
            z = (np.log(merged["T_M"]) - np.log(merged["SOC_raw"])) / merged["sd_lnT"]
            merged["P_below"] = norm.cdf(z)
        out[r] = merged.groupby("Ecoregion_id")["P_below"].mean()
    return pd.DataFrame(out)


P_empirical = compute_ecoregion_P()
P_lit_20 = compute_ecoregion_P(20.0)
P_lit_11 = compute_ecoregion_P(11.0)
print("Ecoregion-mean P(SOC<T), first rows, empirical thresholds:")
P_empirical.head()

Ecoregion-mean P(SOC<T), first rows, empirical thresholds:


,BD,pH,CEC,NPP
Ecoregion_id,,,,
1,0.433008,0.421098,0.650247,0.703511
2,0.642118,0.770181,0.375784,0.463251
3,0.457385,0.562382,0.619303,0.587336
4,0.467069,0.774763,0.663532,0.628736
5,0.588654,0.683484,0.674222,0.790117


In [10]:
hist_lookup = hist_eco.set_index("Ecoregion_id")[["D_2010_pct", "area_km2"]]
D_MEDIAN = hist_eco["D_2010_pct"].median()


def boundary_status(P_df):
    df = P_df.join(hist_lookup, how="inner")
    n_domains_below = (P_df.loc[df.index] >= 0.5).sum(axis=1)
    ev1 = (n_domains_below >= 2).astype(int)
    ev2 = (df["D_2010_pct"] >= D_MEDIAN).astype(int)
    ev3 = (P_df.loc[df.index].mean(axis=1) >= 0.5).astype(int)
    count = ev1 + ev2 + ev3
    status = np.select([count == 3, count == 2], ["BEYOND BOUNDARY", "APPROACHING BOUNDARY"], default="WITHIN BOUNDS")
    return pd.DataFrame({"Ecoregion_id": df.index, "evidence_count": count.values, "status": status, "area_km2": df["area_km2"].values})


boundary_results = {}
for label, P_df in [("Empirical (tiered) thresholds", P_empirical),
                     ("Literature: 2.0% SOC", P_lit_20),
                     ("Literature: 1.1% SOC", P_lit_11)]:
    b = boundary_status(P_df)
    area_pct = 100 * b.groupby("status")["area_km2"].sum() / b["area_km2"].sum()
    boundary_results[label] = area_pct
    print(f"\n{label}:")
    print(area_pct.round(1).to_string())

boundary_comparison = pd.DataFrame(boundary_results).T
boundary_comparison.to_csv(f"{OUT_DIR}/boundary_definition_comparison.csv")
boundary_comparison


Empirical (tiered) thresholds:
status
APPROACHING BOUNDARY    12.5
BEYOND BOUNDARY         60.4
WITHIN BOUNDS           27.1

Literature: 2.0% SOC:
status
APPROACHING BOUNDARY     7.9
BEYOND BOUNDARY         73.4
WITHIN BOUNDS           18.7

Literature: 1.1% SOC:
status
APPROACHING BOUNDARY     5.1
BEYOND BOUNDARY         36.4
WITHIN BOUNDS           58.6


status,APPROACHING BOUNDARY,BEYOND BOUNDARY,WITHIN BOUNDS
Empirical (tiered) thresholds,12.499207,60.377561,27.123232
Literature: 2.0% SOC,7.938740,73.396087,18.665173
Literature: 1.1% SOC,5.064904,36.376169,58.558927


In [11]:
fig, ax = plt.subplots(figsize=(8, 5))
STATE_COLORS = {"WITHIN BOUNDS": "#2e7d32", "APPROACHING BOUNDARY": "#f9a825", "BEYOND BOUNDARY": "#7b0000"}
states = ["WITHIN BOUNDS", "APPROACHING BOUNDARY", "BEYOND BOUNDARY"]
bottom = np.zeros(len(boundary_comparison))
for s in states:
    vals = boundary_comparison[s].values if s in boundary_comparison.columns else np.zeros(len(boundary_comparison))
    ax.bar(boundary_comparison.index, vals, bottom=bottom, color=STATE_COLORS[s], label=s)
    bottom += vals
ax.set_ylabel("% of India's classified land area")
ax.set_title("Degradation boundary classification: empirical vs. literature-cited thresholds")
ax.legend(fontsize=8)
plt.xticks(rotation=10, ha="right")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig_boundary_definition_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Historical depletion vs. contemporary functional risk

In [12]:
def correlation_table(P_df, label):
    rows = []
    for r in RESPONSES:
        merged = pd.DataFrame({"P_below": P_df[r]}).join(hist_lookup, how="inner").dropna()
        rr, pp = stats.pearsonr(merged["D_2010_pct"], merged["P_below"])
        rows.append({"Threshold basis": label, "Response": RESPONSE_FULL[r], "n": len(merged),
                     "Pearson_r": rr, "p_value": pp, "R2": rr ** 2})
    return pd.DataFrame(rows)


corr_all = pd.concat([
    correlation_table(P_empirical, "Empirical (tiered)"),
    correlation_table(P_lit_20, "Literature 2.0% SOC"),
    correlation_table(P_lit_11, "Literature 1.1% SOC"),
], ignore_index=True)
corr_all.to_csv(f"{OUT_DIR}/historical_risk_correlation_comparison.csv", index=False)
corr_all

,Threshold basis,Response,n,Pearson_r,p_value,R2
0,Empirical (tiered),Physical (Bulk Density),48,0.083496,0.572606,0.006972
1,Empirical (tiered),Chemical (pH),48,0.465221,0.000863,0.216430
2,Empirical (tiered),Chemical (CEC),48,0.476433,0.000619,0.226988
3,Empirical (tiered),Ecosystem (NPP),48,0.221440,0.130384,0.049036
4,Literature 2.0% SOC,Physical (Bulk Density),48,0.593155,0.000009,0.351833
5,Literature 2.0% SOC,Chemical (pH),48,0.593155,0.000009,0.351833
6,Literature 2.0% SOC,Chemical (CEC),48,0.593155,0.000009,0.351833
7,Literature 2.0% SOC,Ecosystem (NPP),48,0.593155,0.000009,0.351833
8,Literature 1.1% SOC,Physical (Bulk Density),48,0.375841,0.008474,0.141257
9,Literature 1.1% SOC,Chemical (pH),48,0.375841,0.008474,0.141257


## 7. Sensitivity and resilience: pre-computed summary

In [13]:
sensitivity_summary = pd.read_csv(f"{DATA_DIR}/step19_sensitivity_summary.csv")
print(sensitivity_summary.to_string(index=False))

                                                                      Variant  pct_unchanged_status  spearman_rho  area_weighted_pct_beyond  unit_weighted_pct_beyond
                                 S1: GAM instead of Segmented threshold model             44.897959      0.784371                       NaN                       NaN
                                       S2a: Optimistic (T_L, lower threshold)             42.857143      0.726889                       NaN                       NaN
                                      S2b: Pessimistic (T_U, upper threshold)             53.061224      0.724903                       NaN                       NaN
                              S3: High-support-only minimum-support criterion             44.897959      0.829146                       NaN                       NaN
                           S4: Area-weighted vs Ecoregion-weighted (% Beyond)                   NaN           NaN                 63.532708                 34.693878
    

In [14]:
resilience_a = pd.read_csv(f"{DATA_DIR}/step18_group_a_resilience_opportunity.csv")
resilience_b = pd.read_csv(f"{DATA_DIR}/step18_group_b_continued_loss_risk.csv")
print(f"Resilience opportunity: {len(resilience_a)} ecoregions, {resilience_a['Area_km2'].sum():,.0f} km2")
print(f"Continued-loss risk: {len(resilience_b)} ecoregions, {resilience_b['Area_km2'].sum():,.0f} km2")

Resilience opportunity: 14 ecoregions, 1,387,080 km2
Continued-loss risk: 7 ecoregions, 554,632 km2


## 8. Output filenames


In [15]:
from PIL import Image, ImageDraw, ImageFont

panel_a = Image.open(f"{OUT_DIR}/fig_literature_threshold_comparison.png").convert("RGB")
panel_b = Image.open(f"{OUT_DIR}/fig_boundary_definition_comparison.png").convert("RGB")
h = max(panel_a.height, panel_b.height)
gap = 30
canvas = Image.new("RGB", (panel_a.width + panel_b.width + gap, h), "white")
canvas.paste(panel_a, (0, 0))
canvas.paste(panel_b, (panel_a.width + gap, 0))
draw = ImageDraw.Draw(canvas)
try:
    font = ImageFont.truetype("/System/Library/Fonts/Helvetica.ttc", 42)
except Exception:
    font = ImageFont.load_default()
draw.text((10, 10), "(a)", fill="black", font=font)
draw.text((panel_a.width + gap + 10, 10), "(b)", fill="black", font=font)
canvas.save(f"{OUT_DIR}/Fig9_literature_thresholds.png")

threshold_comparison.to_csv(f"{OUT_DIR}/tab_littresh.csv", index=False)
boundary_comparison.to_csv(f"{OUT_DIR}/tab_littresh_boundary.csv")
corr_all.to_csv(f"{OUT_DIR}/tab_littresh_corr.csv", index=False)
boundary_comparison.to_csv(f"{OUT_DIR}/tabS_littresh_boundary_supp.csv")
corr_all.to_csv(f"{OUT_DIR}/tabS_littresh_corr_supp.csv", index=False)

print("Saved Fig9_literature_thresholds.png, tab_littresh.csv, tab_littresh_boundary.csv, "
      "tab_littresh_corr.csv, tabS_littresh_boundary_supp.csv, tabS_littresh_corr_supp.csv")


Saved Fig9_literature_thresholds.png, tab_littresh.csv, tab_littresh_boundary.csv, tab_littresh_corr.csv, tabS_littresh_boundary_supp.csv, tabS_littresh_corr_supp.csv


## 9. Summary outputs

In [16]:
print("Files written to outputs/:")
for f in sorted(os.listdir(OUT_DIR)):
    print(" ", f)

Files written to outputs/:
  Fig1_spatial_maps.png
  Fig2_threshold_hierarchy.png
  Fig3_uncertainty.png
  Fig4_functional_states.png
  Fig5_2d_framework.png
  Fig6_matrix.png
  Fig7_boundary.png
  Fig8_resilience.png
  Fig9_literature_thresholds.png
  FigS10_hotspots.png
  FigS11_sensitivity.png
  FigS12_scorecard.png
  FigS4_trajectory.png
  FigS6_predicttest.png
  FigS7_retention.png
  FigS8_proximity.png
  FigS9_weighting.png
  boundary_definition_comparison.csv
  fig_boundary_definition_comparison.png
  fig_literature_threshold_comparison.png
  historical_baseline_ecoregion.csv
  historical_risk_correlation_comparison.csv
  literature_threshold_comparison.csv
  tabS_directional.csv
  tabS_hotspots_full.csv
  tabS_littresh_boundary_supp.csv
  tabS_littresh_corr_supp.csv
  tabS_sens_full.csv
  tabS_weighting_full.csv
  tab_boundarycriteria.csv
  tab_convergence.csv
  tab_littresh.csv
  tab_littresh_boundary.csv
  tab_littresh_corr.csv
  tab_resilience.csv
  tab_sensitivity_boundary.